# FlyEyeSimulator — Interactive Demo

Build a Retina + Screen, run Static and Active FES on a saccade video,
visualise both stages, convert photon rates → photoreceptor voltage with
`phototransduction`, then show everything in one synchronised figure
with `IOViz`.


# 0. Imports & paths

In [ ]:
from pathlib import Path
import sys

# Make flyeyesimulator importable from a checkout (no pip install needed).
_THIS_DIR  = Path.cwd()
_REPO_ROOT = _THIS_DIR.parent.parent          # docs/demos -> docs -> repo root
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import time
import numpy as np
import matplotlib.pyplot as plt
import scipy.io

import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook'   # inline Plotly in JupyterLab AND classic Jupyter

from flyeyesimulator import (
    Retina, Screen,
    FlyEyeSimulatorStatic, FlyEyeSimulatorActive,
    SimulatorViz, ScreenViz, RetinaViz,
    xp, to_numpy, BACKEND,
)

COL_JSON = _REPO_ROOT / 'neurocircuitdesk' / 'libs' / 'jsons' / 'hexcol_l1m3_new_578.json'
DATA_DIR = _THIS_DIR  / 'data'
OUT_DIR  = _THIS_DIR  / 'outputs' / 'fes_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'flyeyesimulator backend: {BACKEND}')

# I. Build a single retina

20-ring hex spiral, neural superposition wired at construction,
masked to the biological column set by `hexcol_l1m3_new_578.json`.


In [ ]:
my_retina = Retina(
    num_rings=20,
    inter_ommatidia_angle_deg=4.4,
    bio_cols_only=True,
    col_json_path=str(COL_JSON),
)
n_vrfs = len(my_retina.vrfs)


# II. Visualise the retina and screen

`SimulatorViz` renders the retina as hex-prism lenses and (optionally) the
spherical screen, in an interactive 3D Plotly scene. Pan / zoom / hover to
inspect. We show three views below: bare geometry, an R-cell ray fan, and
all seven R-axes through column 0.

### Bare geometry — retina + screen, no rays

In [ ]:
viz = SimulatorViz(lens_radius=0.04, lens_length=0.01,
                   rays=False, ray_mode='R7', ray_length=8, ray_width=2)
viz.add_retina(my_retina, color="steelblue", name="eye", ray_color='blue')
viz.add_screen(radius=10)
viz.plot()

### Ray fan — neural superposition

R1 of column 5, R2 of column 6, R3 of column 18, R4 of column 1, R5 of
column 2, R6 of column 3 all point at the *same* world location (column 0)
thanks to neural superposition. The fan below makes that visually concrete.

In [ ]:
viz = SimulatorViz(lens_radius=0.04, lens_length=0.01,
                   rays=False, ray_mode='R7', ray_length=8, ray_width=2)
viz.add_retina(my_retina, color="steelblue", name="eye", ray_color='blue')
viz.add_screen(radius=10)
viz.add_rays([
    ('eye', 0,  'R7', 'red'),     # R7 stays along the column's own axis
    ('eye', 5,  'R1', 'blue'),
    ('eye', 6,  'R2', 'blue'),
    ('eye', 18, 'R3', 'blue'),
    ('eye', 1,  'R4', 'blue'),
    ('eye', 2,  'R5', 'blue'),
    ('eye', 3,  'R6', 'blue'),
])
viz.plot()

### All seven R-axes of column 0

In [ ]:
viz = SimulatorViz(lens_radius=0.04, lens_length=0.01,
                   rays=False, ray_mode='R7', ray_length=8, ray_width=2)
viz.add_retina(my_retina, color="steelblue", name="eye", ray_color='blue')
viz.add_screen(radius=10)
viz.add_rays([
    ('eye', 0, r_type, 'red') for r_type in ('R7', 'R1', 'R2', 'R3', 'R4', 'R5', 'R6')
])
viz.plot()

# III. Build the saccade input video

A 300-frame horizontal saccade over `image1.mat`, 256×256 per frame.


In [ ]:
def vid_gen_saccade(bg_img: np.ndarray, *,
                    T: int = 300, H: int = 256, W: int = 256,
                    y_start: int = None, x_start: int = None,
                    speed: float = 4.0,
                    vertical: bool = False, reverse: bool = True) -> np.ndarray:
    """Slide an (H, W) window across ``bg_img`` for ``T`` frames.

    Mirrors the spirit of the reference notebook's ``vid_gen_custom``: a
    pure horizontal (or vertical) saccade with no foreground dots, suitable
    for driving FES with a moving natural scene.

    Returns (T, H, W) float32 in [0, 1].
    """
    bg_h, bg_w = bg_img.shape
    if y_start is None:
        y_start = (bg_h - H) // 2
    if x_start is None:
        x_start = (bg_w - W) if reverse else 0
    direction = -1 if reverse else 1

    video = np.zeros((T, H, W), dtype=np.float32)
    for t in range(T):
        off = int(round(t * speed * direction))
        y0 = (y_start + off) if vertical else y_start
        x0 = (x_start + off) if not vertical else x_start

        sy0, sx0 = max(0, y0), max(0, x0)
        sy1, sx1 = min(bg_h, y0 + H), min(bg_w, x0 + W)
        if sy1 > sy0 and sx1 > sx0:
            dy0, dx0 = sy0 - y0, sx0 - x0
            video[t, dy0:dy0 + (sy1 - sy0),
                    dx0:dx0 + (sx1 - sx0)] = bg_img[sy0:sy1, sx0:sx1]
    return video

In [ ]:
# Load natural-scene image and normalise to [0, 1].
img = np.array(scipy.io.loadmat(str(DATA_DIR / 'image1.mat'))['im'],
               dtype=np.float64)[325:725]
img = (img - img.min()) / (img.max() - img.min())

T, H, W = 200, 256, 256
x0 = vid_gen_saccade(img, T=T, H=H, W=W, speed=4.0,
                     vertical=False, reverse=True)


# IV. Project the video onto a spherical `Screen`

Albers-like rasterisation onto an (elevation × azimuth) hemisphere.
Scale to photon-rate range first (~3·10⁵).


In [ ]:
photon_video = 3e5 * x0 / x0.max()
photon_arr   = xp.asarray(photon_video)
screen = Screen(photon_arr, radius=10, parallels=H, meridians=W, dt=10)


In [ ]:
# Visualise one frame of the projection on the hemisphere
screen_viz = ScreenViz(screen)
screen_viz.plot(frame_idx=T // 2, downsample=2)

# V. Run `FlyEyeSimulatorStatic`

Fixed receptive fields. The cheapest baseline.


In [ ]:
fes_static = FlyEyeSimulatorStatic(
    screen=screen, retina=my_retina, acceptance_angle_deg=4.4)
res_static = fes_static.run(release=False)   # release=False keeps screen alive
res_static_np = to_numpy(res_static)


# VI. Run `FlyEyeSimulatorActive`

Same scene, same retina, but spring-damper RF dynamics update the
acceptance angle and elevation every step.


In [ ]:
# Independent screen: the active simulator releases its screen at end of run.
screen_active = Screen(photon_arr, radius=10, parallels=H, meridians=W, dt=10)

fes_active = FlyEyeSimulatorActive(
    screen=screen_active, retina=my_retina,
    initial_acceptance_angle_deg=4.4,
    shift_ratio=1, shrink_ratio=0.5,
)
res_active = fes_active.run(release=False)
res_active_np = to_numpy(res_active)


# VII. Compare static vs active responses


In [ ]:
# Spatial snapshot at one mid-saccade frame
frame_idx = T // 2
ret_viz = RetinaViz(my_retina, lens_radius=0.045, lens_length=0.01)
ret_viz.plot(res_static_np[:,::6][frame_idx], title=f'Static FES @ t={frame_idx}')
ret_viz.plot(res_active_np[:,::6][frame_idx], title=f'Active FES @ t={frame_idx}')

In [ ]:
# R1 channel time series for a few columns
n_cols = n_vrfs // 6
sample_cols = [0, n_cols // 8, n_cols // 4, n_cols // 2]

fig, axes = plt.subplots(len(sample_cols), 1,
                         figsize=(10, 2.5 * len(sample_cols)),
                         sharex=True)
for ax, col in zip(axes, sample_cols):
    r1_idx = col * 6
    ax.plot(res_static_np[:, r1_idx], label='Static', alpha=0.85, lw=1.2)
    ax.plot(res_active_np[:, r1_idx], label='Active', alpha=0.85, lw=1.2)
    ax.set_ylabel(f'Col {col}\nR1 photon rate')
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Frame')
fig.suptitle('Static vs Active sampling — R1 photon rate over the saccade')
fig.tight_layout()
plt.show()


### Side-by-side video — screen, static, active

Independent `ScreenViz.save_video` + `RetinaViz.save_video_row` (two
animations side-by-side, not synchronised). The next section (§VII.5)
combines them into one synchronised figure with `IOViz`.


In [ ]:
fig_screen = screen_viz.save_video(fps=15, downsample=4,
                                   html_path=str(OUT_DIR / 'screen.html'))
display(fig_screen)

ret_viz_row = RetinaViz(my_retina, lens_radius=0.04, lens_length=0.01)
ret_viz_row.save_video_row(
    [res_static_np[:, ::6], res_active_np[:, ::6]],
    ['Static FES', 'Active FES'],
    fps=15, dt_ms=10,
    html_path=str(OUT_DIR / 'static_active.html'),
)


# VII.5 Phototransduction — photon rates → photoreceptor voltage

`pr(...)` runs the RPM cascade + non-spiking HH membrane on the `(T, 6·N_cols)`
photon-rate array. `aggregate_pr(...)` averages the 6 R-cells in
neural-superposition for each target column via `superposition.get_prs`.
Defaults assume 100 fps (`itpl_val=1000, dt=1e-5`); backend matches FES's
`BACKEND` automatically.


In [ ]:
from flyeyesimulator import pr, aggregate_pr

n_cols = n_vrfs // 6
V_static_TN6 = pr(res_static_np)                      # (T, 6·N_cols) voltage in mV

panel_data_TN = [
    res_static_np[:, ::6],                            # R1 photon rate (input)
    V_static_TN6[:, ::6],                             # R1 voltage    (output)
    aggregate_pr(V_static_TN6,  num_cols=n_cols),     # R1–R6 avg voltage
]
panel_titles = ['R1 photon (in)', 'R1 voltage (out)', 'R1–R6 avg voltage (out)']

### Combined view — screen + R1 in/out + R1–R6 avg voltage

One synchronised Plotly figure built via `IOViz`: spherical screen on
the far left, then three retina meshes — R1 photon rate (input), R1
voltage (output), R1–R6 superposition-averaged voltage (output). One
time slider, one Play/Pause control, per-panel colour ranges and
colorbars. `IOViz` reuses `RetinaViz` and `Screen` internals — no
manual mesh assembly here.


In [ ]:
from flyeyesimulator import IOViz

# IOViz builds a synchronised screen + retina-panel video. The constructor
# reuses RetinaViz.get_r1_mesh() and Screen.get_sphere_geometry() — no
# manual mesh assembly. The screen lives on the far left; the three retina
# panels share one time slider and Play/Pause control.
#
# screen_downsample=4 (default) reduces the 256x256 hemisphere to 64x64 so
# the figure fits inline in a browser; frame_stride=5 keeps every 5th frame
# of the 300-frame run (60 effective frames, slider in 50 ms steps). Drop
# both to 1 for a full-resolution HTML export.
io_viz = IOViz(
    screen=screen,
    retina_viz=ret_viz,                              # built earlier in cell 24
    colorscale='Viridis',                            # retina panels
    screen_colorscale='gray',                        # screen
    screen_downsample=4,
)

# fig = io_viz.save_video(
#     retina_values=panel_data_TN,                     # built in §VII.5
#     titles=panel_titles,
#     screen_crange=(0, 3e5),                          # photon-rate scale
#     retina_crange=[
#         None,            # R1 photon  — auto
#         (-80, 0),        # R1 voltage — clamp to plausible mV range
#         (-80, 0),        # avg voltage
#     ],
#     fps=15,
#     dt_ms=10,
#     frame_stride=5,
#     html_path=str(OUT_DIR / 'screen_pt_combined.html'),
# )
# display(fig)


### Combined view (2-row) — static vs active sampling

Same `IOViz` instance, **two-row** layout (`retina_values` is a nested
list ⇒ screen `rowspan=2` on the left, with the three panels for
**static** sampling on top and **active** sampling on the bottom).
One synchronised time slider scrubs all seven scenes together.


In [ ]:
V_active_TN6 = pr(res_active_np)                       # (T, 6·N_cols) voltage, mV

panel_data_TN_active = [
    res_active_np[:, ::6],                             # R1 photon rate
    V_active_TN6[:, ::6],                              # R1 voltage
    aggregate_pr(V_active_TN6, num_cols=n_cols),       # R1–R6 avg voltage
]

fig2 = io_viz.save_video(
    retina_values=[panel_data_TN, panel_data_TN_active],   # nested ⇒ 2-row
    titles=[
        ['Static R1 photon', 'Static R1 voltage', 'Static R1–R6 avg V'],
        ['Active R1 photon', 'Active R1 voltage', 'Active R1–R6 avg V'],
    ],
    screen_crange=(0, 3e5),
    retina_crange=[
        [None, (-80, 0), (-80, 0)],
        [None, (-80, 0), (-80, 0)],
    ],
    fps=15,
    dt_ms=10,
    frame_stride=5,
    html_path=str(OUT_DIR / 'screen_pt_combined_2rows.html'),
)
display(fig2)


# VIII. (Optional) Persist the outputs


In [ ]:
import h5py
with h5py.File(OUT_DIR / 'fsaccade_lambda.h5', 'w') as f:
    f['video']        = x0.astype(np.float32)
    f['photon_rate']  = photon_video.astype(np.float32)
    f['res_static']   = res_static_np
    f['res_active']   = res_active_np
    f['V_static']     = V_static_TN6.astype(np.float32)
    f['V_static_R1']  = panel_data_TN[1].astype(np.float32)
    f['V_static_avg'] = panel_data_TN[2].astype(np.float32)
    f['V_active']     = V_active_TN6.astype(np.float32)
    f['V_active_R1']  = panel_data_TN_active[1].astype(np.float32)
    f['V_active_avg'] = panel_data_TN_active[2].astype(np.float32)


---

### Next steps

- Swap the **single retina** for a left/right pair (`RetinaRotator` with a
  mirror + Euler rotation) and re-run — see the `dual_pick` demo in
  `demo_flyeye.py`.
- Feed the **`R1–R6 avg voltage`** trace from §VII.5 into the looming or
  motion NCD circuits (`demo_looming.ipynb` / `demo_motion_circuit.py`).
  That's the per-column photoreceptor voltage NCD circuits typically
  expect at their `input_main` port.
- Replace the saccade with a looming-dot stimulus (see the
  `make_looming_dot_video` helper in `demo_active_sampling.py`).
